### Setup

In [ ]:
%pip install uv

In [ ]:
!uv pip install -U mlx-vlm datasets pillow matplotlib seaborn

In [ ]:
import json
import re
import sys
from pathlib import Path
from typing import Any, cast
from collections import Counter
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from curses import raw

import gc
from datasets import load_dataset
from mlx_vlm.generate import generate
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config, load
import mlx.core as mx


## Step 1.1 - Normalise dataset's annotation format

Multi-source OCR and receipt datasets mix conflicting regional numerical conventions:

US Format: 1,234.56 <br>
EU Format: 1.234,56 <br>
Integer-Only Format: 60.000 (e.g., IDR)

Implementation: All price annotations are pre-processed into the standardized US system (1234.56 / standard float) before model ingestion. <br>
Enforcing a single representation removes schema ambiguity, prevents silent float-parsing errors, and improves fine-tuning convergence on Qwen2-VL.

Dataset format:
{
    "items": [
        {
            "name": "...", 
            "price": 12.5, 
            "quantity": 1
        }
    ],
    "total": 12.5
}

In [ ]:
def normalize_price(raw):
    if raw is None:
        return None

    # Remove any non-numeric characters except for decimal separators
    s = str(raw).strip()
    if not s:
        return None

    # Remove any non-numeric characters except for decimal separators
    s = re.sub(r"[^\d.,]", "", s)
    if not s:
        return None

    # Determine the last separator and how many digits follow it
    seps = list(re.finditer(r"[.,]", s))
    if not seps:
        try:
            return round(float(s), 2)
        except ValueError:
            return None

    # If the last separator is followed by exactly 3 digits, 
    # it's likely a thousands separator; otherwise, it's a decimal separator.
    last_sep_pos = seps[-1].start()
    digits_after_last_sep = len(s) - last_sep_pos - 1
    
    if digits_after_last_sep == 3:
        s = re.sub(r"[.,]", "", s)
    else:
        before, after = s[:last_sep_pos], s[last_sep_pos + 1:]
        before = re.sub(r"[.,]", "", before)
        s = f"{before}.{after}" if after else before

    try:
        return round(float(s), 2)
    except ValueError:
        return None

def _as_list(value):
    # Helper function to ensure the value is always returned as a list.
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

# CORD schema to target format conversion
def cord_to_target(gt_parse):
    items = []
    for entry in _as_list(gt_parse.get("menu")):
        if not isinstance(entry, dict):
            continue
        name = (entry.get("nm") or entry.get("name") or "").strip()
        if not name:
            continue
        price = normalize_price(entry.get("price") or entry.get("unitprice"))
        qty_raw = entry.get("cnt") or entry.get("count") or "1"
        try:
            quantity = max(1, int(re.sub(r"[^\d]", "", str(qty_raw)) or "1"))
        except ValueError:
            quantity = 1
        if price is None:
            continue
        items.append({"name": name, "price": price, "quantity": quantity})

    total = None
    total_block = gt_parse.get("total")
    if isinstance(total_block, dict):
        total = normalize_price(total_block.get("total_price"))
    if total is None and items:
        total = round(sum(i["price"] * i["quantity"] for i in items), 2)

    return {"items": items, "total": total}


def parse_cord_example(ground_truth_json_str):
    try:
        outer = json.loads(ground_truth_json_str)
        gt_parse = outer.get("gt_parse", outer)
        return cord_to_target(gt_parse)
    except (json.JSONDecodeError, AttributeError):
        return None

## Step 1.2 - Test CORD dataset format conversion

The CORD (Consolidated Receipt Dataset) dataset uses a complex, deeply nested JSON hierarchy built around OCR bounding boxes and custom tags.

menu.nm, menu.price, total.creditcardprice

Vision-Language Models cannot be trained on this raw structure directly; they require a clean instruction-following format where the visual image pairs with a single, valid JSON text target

In [ ]:
# Test cases for the parsing and normalization functions
_multi = json.dumps({"gt_parse": {
    "menu": [{"nm": "NASI GORENG", "cnt": "1", "price": "25,000"},
             {"nm": "ES TEH", "cnt": "2", "price": "10,000"}],
    "total": {"total_price": "35,000"},
}})
_single = json.dumps({"gt_parse": {
    "menu": {"nm": "AYAM BAKAR", "cnt": "1", "price": "18,000"},
    "total": {"total_price": "18,000"},
}})

# Test the parsing and normalization functions
_r = parse_cord_example(_multi)
assert _r is not None
assert _r["items"] == [
    {"name": "NASI GORENG", "price": 25000.0, "quantity": 1},
    {"name": "ES TEH", "price": 10000.0, "quantity": 2},
], _r
assert _r["total"] == 35000.0

# Dict-shaped menu, the CORD quirk
_r = parse_cord_example(_single)  
assert _r is not None
assert _r["items"] == [{"name": "AYAM BAKAR", "price": 18000.0, "quantity": 1}], _r

# Test invalid JSON and price normalization
assert parse_cord_example("{not valid json") is None
assert normalize_price("10,000") == 10000.0
assert normalize_price("$12.50") == 12.50
assert normalize_price(None) is None

print("All schema tests passed")

## Step 2 - CORD Dataset Ingestion

This step processes transformed CORD-v2 receipt records into standard JSON Lines (.jsonl) manifests formatted specifically for mlx-vlm fine-tuning with Qwen2-VL. It creates the exact directory layout required to avoid Hugging Face dataset loader conflicts.

In [ ]:
SYSTEM_PROMPT = (
    "Extract every line item from this receipt. Respond with ONLY a JSON "
    "object in exactly this schema, no other text:\n"
    '{"items": [{"name": string, "price": number, "quantity": number}], "total": number}'
)

MANIFEST_DIR = Path("./data")       # pass THIS as --dataset
IMAGES_DIR = Path("./data_images")

# Export a split to the manifest and image directories
def export_split(split, split_name: str) -> Path:
    images_dir = IMAGES_DIR / split_name
    images_dir.mkdir(parents=True, exist_ok=True)
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = MANIFEST_DIR / f"{split_name}.jsonl"
    
    written, skipped = 0, 0
    with open(manifest_path, "w") as f:
        for i, sample in enumerate(split):
            # Parse the ground truth JSON to extract items and total
            target = parse_cord_example(sample["ground_truth"])

            # Skip samples that are malformed or have no items
            if target is None or not target["items"]:
                skipped += 1
                continue

            image_path = images_dir / f"{i:05d}.png"
            sample["image"].convert("RGB").save(image_path)

            record = {
                "images": [str(image_path)],
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": SYSTEM_PROMPT},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [{"type": "text", "text": json.dumps(target, ensure_ascii=False)}],
                    },
                ],
            }
            f.write(json.dumps(record) + "\n")
            written += 1

    print(f"{split_name}: wrote {written} examples, skipped {skipped} malformed")
    return manifest_path

In [ ]:
dataset = load_dataset("naver-clova-ix/cord-v2")

# Export each split (train, validation, test) to the manifest and image directories
# Validation split is renamed to "valid" to match mlx-examples-style trainers' expectations
SPLIT_MAP = {
    "train": "train",
    "validation": "valid",
    "test": "test",
}

for hf_split, export_name in SPLIT_MAP.items():
    if hf_split in dataset:
        export_split(dataset[hf_split], export_name)

## Step 3 - Test Evaluation Methods

This step unit-tests and validates the custom evaluation framework before running full dataset benchmarks, ensuring that scoring metrics operate deterministically across diverse model outputs. By executing sanity checks against synthetic test vectors—including perfect matches, key aliases, malformed JSON, and plain-text item arrays—we verify that repair_json(), canonicalize_prediction(), and score_prediction() correctly normalize predictions and calculate:

multiset precision, recall, and F1-scores via Counter. 

Thoroughly testing this evaluation harness independently guarantees that subsequent zero-shot baseline and fine-tuned model comparisons yield true, unbiased performance metrics rather than metric calculation or post-processing errors.

In [ ]:
def repair_json(text):
    if not text or not isinstance(text, str):
        return None

    text = text.strip()

    # Strip Markdown code block fences (e.g., ```json ... ``` or ``` ... ```)
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text).strip()

    # Extract embedded JSON object/array if wrapped in freeform text
    match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if match:
        text = match.group(1).strip()

    # Direct JSON parse attempt
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Clean trailing commas (e.g. {"a": 1,})
    repaired = re.sub(r",\s*([\]}])", r"\1", text)

    # Auto-close unclosed brackets/braces
    repaired += "]" * max(0, repaired.count("[") - repaired.count("]"))
    repaired += "}" * max(0, repaired.count("{") - repaired.count("}"))

    try:
        return json.loads(repaired)
    except json.JSONDecodeError:
        return None
    
def diff_items(predicted_items, gt_items):
    # Normalize item names and count occurrences for comparison
    pred_counts = Counter(normalize_item(i) for i in predicted_items)
    gt_counts = Counter(normalize_item(i) for i in gt_items)
    missed = gt_counts - pred_counts        # in ground truth, not predicted (hurts recall)
    hallucinated = pred_counts - gt_counts  # in prediction, not ground truth (hurts precision)
    return missed, hallucinated

## Step 4 -  Zero-Shot Baseline Evaluation

Establish an un-tuned performance baseline for chosen model on receipt line-item and total extraction prior to Low-Rank Adaptation (LoRA) fine-tuning. This benchmark provides the baseline metrics needed to quantify exact empirical gains in schema adherence, item recall, and price accuracy after training

In [ ]:
def normalize_item(item):
    # Handle cases where model outputs raw strings instead of dict objects
    if not isinstance(item, dict):
        return (str(item).strip().lower(), 0.0, 1)

    # Extract and normalize the name, price, and quantity
    name = str(item.get("name") or "").strip().lower()

    # Safely convert price to float
    raw_price = item.get("price")
    try:
        price = (
            round(float(raw_price), 2) if raw_price is not None else 0.0
        )
    except (ValueError, TypeError):
        price = 0.0

    # Safely convert quantity to int
    raw_qty = item.get("quantity") or item.get("qty") or 1
    try:
        quantity = int(raw_qty)
    except (ValueError, TypeError):
        quantity = 1

    return (name, price, quantity)


def score_prediction(predicted, ground_truth):
    # If either predicted or ground truth is not a dict, return zero scores and total_correct as False
    if not isinstance(predicted, dict) or not isinstance(ground_truth, dict):
        return {
            "item_precision": 0.0,
            "item_recall": 0.0,
            "item_f1": 0.0,
            "total_correct": False,
        }

    raw_pred_items = predicted.get("items", [])
    raw_gt_items = ground_truth.get("items", [])

    # Ensure items is iterable
    if not isinstance(raw_pred_items, list):
        raw_pred_items = []
    if not isinstance(raw_gt_items, list):
        raw_gt_items = []

    # Normalize items for comparison
    pred_items = [normalize_item(i) for i in raw_pred_items]
    gt_items = [normalize_item(i) for i in raw_gt_items]

    # Count true positives, false positives, and false negatives using Counter
    pred_counts, gt_counts = Counter(pred_items), Counter(gt_items)
    true_positives = sum((pred_counts & gt_counts).values())

    # Calculate precision, recall, and F1 score
    precision = true_positives / len(pred_items) if pred_items else 0.0
    recall = true_positives / len(gt_items) if gt_items else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    pred_total = predicted.get("total")
    gt_total = ground_truth.get("total")

    # Check if the predicted total is approximately equal to the ground truth total
    try:
        total_correct = (
            pred_total is not None
            and gt_total is not None
            and abs(float(pred_total) - float(gt_total)) < 0.01
        )
    except (ValueError, TypeError):
        total_correct = False

    return {
        "item_precision": round(precision, 3),
        "item_recall": round(recall, 3),
        "item_f1": round(f1, 3),
        "total_correct": total_correct,
    }

def score_results(results):
    scores = []
    # Loop through each result, repair the JSON output, 
    # and score the prediction against the ground truth
    for r in results:
        predicted = repair_json(r["output_text"])
        gt = json.loads(r["ground_truth"])
        scores.append(score_prediction(predicted, gt))

    n = len(scores)

    return {
        "n": n,
        "avg_item_precision": round(sum(s["item_precision"] for s in scores) / n, 3),
        "avg_item_recall": round(sum(s["item_recall"] for s in scores) / n, 3),
        "avg_item_f1": round(sum(s["item_f1"] for s in scores) / n, 3),
        "total_accuracy": round(sum(s["total_correct"] for s in scores) / n, 3),
    }

In [ ]:
def get_model_test_output(processor, model, config):
    baseline_results = []
    lines = Path("./data/test.jsonl").read_text().splitlines()
    valid_count = 0

    # Explicit manual progress bar context
    with tqdm(
        total=len(lines), desc="Evaluating", unit="img", file=sys.stdout
    ) as pbar:
        for i, line in enumerate(lines):
            record = json.loads(line)
            image_path = record["images"][0]
            ground_truth = record["messages"][1]["content"][0]["text"]

            formatted_prompt = apply_chat_template(
                processor, config, SYSTEM_PROMPT, num_images=1
            )

            raw_output = generate(
                model,
                processor,
                image=image_path,
                prompt=cast(str, formatted_prompt),
                verbose=False,
            )

            output_text = (
                raw_output.text
                if hasattr(raw_output, "text")
                else str(raw_output)
            )
            repair_output = repair_json(output_text)
            is_valid = isinstance(repair_output, dict)

            if is_valid:
                valid_count += 1

            baseline_results.append({
                "image_path": image_path,
                "output_text": output_text,
                "repaired_json": repair_output,
                "ground_truth": ground_truth,
            })

            # Force increment and UI flush on every loop iteration
            pbar.set_postfix(
                valid=f"{valid_count}/{i + 1}",
                acc=f"{(valid_count / (i + 1)):.0%}",
            )
            pbar.update(1)
            pbar.refresh()

    return baseline_results

In [ ]:
MODEL_2_ID = "mlx-community/Qwen2-VL-2B-Instruct-4bit"
MODEL_3_ID = "mlx-community/Qwen3-VL-2B-Instruct-4bit"

print(f"Loading {MODEL_2_ID}")
model2, processor2 = load(MODEL_2_ID)
config2 = load_config(MODEL_2_ID)

print(f"Loading {MODEL_3_ID}")
model3, processor3 = load(MODEL_3_ID)
config3 = load_config(MODEL_3_ID)

In [ ]:
baseline_results2 = get_model_test_output(processor2, model2, config2)
baseline_results3 = get_model_test_output(processor3, model3, config3)

In [ ]:
baseline_scores2 = score_results(baseline_results2)
baseline_scores3 = score_results(baseline_results3)

print("Zero-shot baseline scores for Qwen2-VL:", baseline_scores2)
print("Zero-shot baseline scores for Qwen3-VL:", baseline_scores3)

## Step 5: Low-Rank Adaptation (LoRA) Fine-Tuning

Execute Parameter-Efficient Fine-Tuning (PEFT) on `Qwen2-VL-2B` natively on Apple Silicon using `mlx-vlm`. Because the base model checkpoint is 4-bit quantized (`-4bit`), `mlx_vlm.lora` automatically applies **QLoRA**—freezing base vision-language weights in 4-bit while updating lightweight LoRA adapters in higher precision.

### Hyperparameter & Parameter Breakdown

| Parameter | Value | Functional Purpose |
| :--- | :--- | :--- |
| `--model` | `mlx-community/Qwen2-VL-2B-Instruct-4bit` | Hugging Face repo path for 4-bit quantized base vision-language model. |
| `--dataset` | `./data` | Directory containing `train.jsonl` and `valid.jsonl` manifests. |
| `--output-path` | `./adapters` | Destination directory for output weights (`adapters.safetensors`) & config. |
| `--iters` | `500` | Total optimization steps to achieve schema alignment without overfitting. |
| `--batch-size` | `1` | Controls memory footprint; keeps peak VRAM usage under ~8 GB. |
| `--learning-rate` | `1e-4` | Learning rate for LoRA adapter weight convergence on quantized base. |

### Understanding Model Quantization: 4-Bit vs. 8-Bit

**Quantization** converts neural network weights from high-precision floating-point numbers (FP16 or FP32) to lower bit-width integers (INT8 or INT4). This drastically reduces VRAM consumption and compute requirements while preserving task performance.

### Technical Comparison

| Metric / Feature | 8-Bit Precision (INT8) | 4-Bit Precision (INT4 / NF4) |
| :--- | :--- | :--- |
| **Memory Footprint** | ~1 byte per parameter (~50% of FP16) | ~0.5 bytes per parameter (~75% of FP16) |
| **Baseline Accuracy** | Near-lossless (< 1% drop vs. FP16) | Slight baseline drop (1–3%), easily recovered via fine-tuning |
| **VRAM Efficiency** | Moderate memory savings | Maximum memory savings |
| **Optimal Use Case** | High-precision production serving | Local Apple Silicon execution & **QLoRA** fine-tuning |

In [ ]:
def print_gpu_memory():
    active_gb = mx.get_active_memory() / 1e9
    peak_gb = mx.get_peak_memory() / 1e9
    cache_gb = mx.get_cache_memory() / 1e9

    print(f"Active GPU Memory: {active_gb:.2f} GB")
    print(f"Peak GPU Memory:   {peak_gb:.2f} GB")
    print(f"Cache GPU Memory:  {cache_gb:.2f} GB")


print_gpu_memory()

In [ ]:
gc.collect()
mx.clear_cache()

In [ ]:
# Fine-tuning Qwen2-VL
# mlx-vlm does not currently support the validation split, so we only use the training split for LoRA fine-tuning.
!python -m mlx_vlm.lora \
    --model mlx-community/Qwen2-VL-2B-Instruct-4bit \
    --dataset ./data \
    --output-path ./adapters2 \
    --max-seq-length 512 \
    --iters 500 \
    --batch-size 1 \
    --learning-rate 1e-4 \
    --steps-per-report 10

In [ ]:
gc.collect()
mx.clear_cache()
mx.set_cache_limit(0)

In [ ]:
# Fine-tuning Qwen3-VL
# mlx-vlm does not currently support the validation split, so we only use the training split for LoRA fine-tuning.
!python -m mlx_vlm.lora \
    --model mlx-community/Qwen3-VL-2B-Instruct-4bit \
    --dataset ./data \
    --output-path ./adapters3 \
    --iters 500 \
    --batch-size 1 \
    --learning-rate 1e-4  \
    --steps-per-report 10

In [ ]:
finetuned_model2, finetuned_processor2 = load(MODEL_2_ID, adapter_path="./adapters2")
finetuned_config2 = load_config(MODEL_2_ID)

# finetuned_model3, finetuned_processor3 = load(MODEL_3_ID, adapter_path="./adapters3")
# finetuned_config3 = load_config(MODEL_3_ID)

In [ ]:
finetuned_results2 = get_model_test_output(finetuned_processor2, finetuned_model2, finetuned_config2)
finetuned_scores2 = score_results(finetuned_results2)

# finetuned_results3 = get_model_test_output(finetuned_processor3, finetuned_model3, finetuned_config3)
# finetuned_scores3 = score_results(finetuned_results3)

print("Zero-shot baseline for Qwen2-VL: ", baseline_scores2)
print("Fine-tuned for Qwen2-VL:         ", finetuned_scores2)

print("Zero-shot baseline for Qwen3-VL: ", baseline_scores3)
# print("Fine-tuned for Qwen3-VL:         ", finetuned_scores3)

## Step 6 - Visualizing Receipt Parsing Performance: Baseline vs. Fine-Tuned Models

This code generates a publication-quality grouped bar chart using `matplotlib` and `numpy` to compare performance across four core metrics: **Item Precision**, **Item Recall**, **Item F1-Score**, and **Total Accuracy**. It compares zero-shot base models against fine-tuned QLoRA adapters across different model generations (`Qwen2-VL` vs. `Qwen3-VL`).

### Code Mechanics & Architecture

* **Metric Alignment & Extraction:** Converts the baseline and fine-tuned result dictionaries into flat numerical arrays (`baseline_vals` and `finetuned_vals`) aligned to the `metrics` category indices.
* **X-Axis Coordinate Offsets:** Uses `np.arange(len(metrics))` to create integer tick locations and shifts bar positions using `x - width / 2` and `x + width / 2` to place comparative bars side-by-side.
* **Layering & Transparency (`alpha`):**
  * Renders `Qwen2-VL` series using solid hex colors (`#94A3B8` for baseline, `#2563EB` for fine-tuned).
  * Renders `Qwen3-VL` series on top using semi-transparency (`alpha=0.5`) to visually contrast the two model generations.
* **Dynamic Metric Labeling (`add_labels`):** Defines a helper function that iterates over bar objects (`rects`), reads each individual height, and attaches rounded 3-decimal values (`f"{height:.3f}"`) directly above each bar via `ax.annotate()`.
* **Plot Cleanliness & Formatting:** Extends the Y-axis upper limit to `1.15` to preserve headroom for floating labels, applies subtle horizontal gridlines (`alpha=0.4`), and invokes `plt.tight_layout()` to prevent legend clipping.

In [ ]:
# Testing the scoring function with a sample prediction and ground truth
metrics = [
    "Item Precision",
    "Item Recall",
    "Item F1-Score",
    "Total Accuracy",
]

# Prepare the values for plotting
baseline_vals2 = [
    baseline_scores2["avg_item_precision"],
    baseline_scores2["avg_item_recall"],
    baseline_scores2["avg_item_f1"],
    baseline_scores2["total_accuracy"],
]
finetuned_vals2 = [
    finetuned_scores2["avg_item_precision"],
    finetuned_scores2["avg_item_recall"],
    finetuned_scores2["avg_item_f1"],
    finetuned_scores2["total_accuracy"],
]

baseline_vals3 = [
    baseline_scores3["avg_item_precision"],
    baseline_scores3["avg_item_recall"],
    baseline_scores3["avg_item_f1"],
    baseline_scores3["total_accuracy"],
]
# finetuned_vals3 = [
#     finetuned_scores3["avg_item_precision"],
#     finetuned_scores3["avg_item_recall"],
#     finetuned_scores3["avg_item_f1"],
#     finetuned_scores3["total_accuracy"],
# ]

x = np.arange(len(metrics))
width = 0.2

fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

rects1 = ax.bar(
    x - 1.5 * width,
    baseline_vals2,
    width,
    label="Zero-Shot Baseline (Qwen2-VL)",
    color="#94A3B8",
)
rects2 = ax.bar(
    x - 0.5 * width,
    finetuned_vals2,
    width,
    label="Fine-Tuned Qwen2-VL",
    color="#2563EB",
)

rects3 = ax.bar(
    x + 0.5 * width,
    baseline_vals3,
    width,
    label="Zero-Shot Baseline (Qwen3-VL)",
    color="#94A3B8",
    alpha=0.5,
)
# rects4 = ax.bar(
#    x + 1.5 * width,
#    finetuned_vals3,
#    width,
#    label="Fine-Tuned Qwen3-VL",
#    color="#2563EB",
#    alpha=0.5,
# )

ax.set_ylabel("Score (0.00 – 1.00)")
ax.set_title(
    "Receipt Parsing Evaluation: Zero-Shot vs. Fine-Tuned Model",
    fontweight="bold",
    pad=15,
)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontweight="semibold")
ax.set_ylim(0, 1.15)
ax.legend(frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.4)


def add_labels(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(
            f"{height:.3f}",
            xy=(rect.get_x() + rect.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )


add_labels(rects1)
add_labels(rects2)
add_labels(rects3)
# add_labels(rects4)

plt.tight_layout()
plt.show()